# Date and Time Data Types and Tools

The Python standard library includes data types for date and time data, as well as calendar-related functionality. The `datetime`, `time`, and `calendar` modules are the main place to start. Everything pandas does with time series — covered in the rest of this chapter — is built on top of these core types, so it's worth getting comfortable with them first.

The `datetime.datetime` type, or simply `datetime`, is the one you'll reach for most: it stores a single instant in time (a **timestamp**), combining a calendar date and a time of day. It's widely used:

In [1]:
import numpy as np 
import pandas as pd

In [2]:
from datetime import datetime 

now = datetime.now()

now

datetime.datetime(2026, 8, 1, 18, 40, 29, 871411)

In [3]:
now.day, now.month, now.year

(1, 8, 2026)

A `datetime` exposes each component as its own attribute, plus a couple of computed conveniences like `weekday()` (Monday = 0):

In [4]:
now.hour, now.minute, now.second, now.microsecond, now.weekday()

(18, 40, 29, 871411, 5)

`datetime` stores both the date and time down to the microsecond. Subtracting two `datetime` objects produces a `datetime.timedelta` (or simply `timedelta`) — a **duration**, as opposed to a point in time:

In [5]:
delta = datetime(2011, 1, 7) - datetime(2008, 6, 24, 8, 15)

delta

datetime.timedelta(days=926, seconds=56700)

In [6]:
delta.days

926

In [7]:
delta.seconds

56700

**Gotcha:** `timedelta` only stores `days`, `seconds`, and `microseconds` as separate attributes — `.seconds` is the *remainder* after whole days are removed (0–86399), **not** the total duration in seconds. If you want the whole thing expressed as one number, use `.total_seconds()`:

In [8]:
delta.total_seconds()

80063100.0

You can add (or subtract) a `timedelta`, or multiple thereof, to a `datetime` object to yield a new, shifted object:

In [9]:
from datetime import timedelta

start = datetime(2011, 1, 7)

start + timedelta(12)

datetime.datetime(2011, 1, 19, 0, 0)

In [10]:
start - 2 * timedelta(12)

datetime.datetime(2010, 12, 14, 0, 0)

## The Core `datetime` Module Types

| Type | Stores |
|---|---|
| `date` | Calendar date (year, month, day) using the Gregorian calendar |
| `time` | Time of day as hours, minutes, seconds, and microseconds |
| `datetime` | Both date and time |
| `timedelta` | The difference between two `datetime` values (as days, seconds, and microseconds) |
| `tzinfo` | Base type for storing time zone information (covered later, in Time Zone Handling) |

## Converting Between String and Datetime

You can format `datetime` objects (and pandas `Timestamp` objects, introduced in the next notebook) as strings using `str` or the `strftime` method, passing a format specification:

In [11]:
stamp = datetime(2011, 1, 3)

str(stamp)

'2011-01-03 00:00:00'

In [12]:
stamp.strftime("%Y-%m-%d")

'2011-01-03'

| Type | Description |
|---|---|
| `%Y` | 4-digit year |
| `%y` | 2-digit year |
| `%m` | 2-digit month [01, 12] |
| `%d` | 2-digit day [01, 31] |
| `%H` | Hour (24-hour clock) [00, 23] |
| `%I` | Hour (12-hour clock) [01, 12] |
| `%M` | 2-digit minute [00, 59] |
| `%S` | Second [00, 61] (60, 61 account for leap seconds) |
| `%w` | Weekday as integer [0, 6], where 0 is Sunday |
| `%U` | Week number of the year [00, 53]; Sunday is the first day of the week; days before the first Sunday are "week 0" |
| `%W` | Week number of the year [00, 53]; Monday is the first day of the week; days before the first Monday are "week 0" |
| `%z` | UTC time zone offset as `+HHMM` or `-HHMM`; empty if time zone naive |
| `%F` | Shortcut for `%Y-%m-%d` (e.g., 2012-04-18) |
| `%D` | Shortcut for `%m/%d/%y` (e.g., 04/18/12) |

You can use many of the same format codes to convert strings to dates using `datetime.strptime` (but some codes, like `%F` cannot be used):

In [13]:
value = "2011-01-03"

datetime.strptime(value, "%Y-%m-%d")

datetime.datetime(2011, 1, 3, 0, 0)

In [14]:
datestrs = ["7/6/2011", "8/6/2011"]

[datetime.strptime(x, "%m/%d/%Y") for x in datestrs]

[datetime.datetime(2011, 7, 6, 0, 0), datetime.datetime(2011, 8, 6, 0, 0)]

## A Smarter Parser: `dateutil`

Writing out a `strftime`/`strptime` format string works, but it's tedious and brittle — every new date format needs its own format string. The third-party `dateutil` package (a pandas dependency, so it's always available) provides a much more flexible parser, `dateutil.parser.parse`, that can handle almost any human-intelligible date representation *without* a format string:

In [15]:
from dateutil.parser import parse

parse("2011-01-03")

datetime.datetime(2011, 1, 3, 0, 0)

In [16]:
parse("Jan 31, 1997 10:45 PM")

datetime.datetime(1997, 1, 31, 22, 45)

**Gotcha:** many countries outside the US write dates day-first (`6/12/2011` meaning 6 December, not June 12th). `dateutil` defaults to month-first; pass `dayfirst=True` when your data comes from a day-first locale:

In [17]:
parse("6/12/2011"), parse("6/12/2011", dayfirst=True)

(datetime.datetime(2011, 6, 12, 0, 0), datetime.datetime(2011, 12, 6, 0, 0))

`datetime.strptime` (and `dateutil.parser.parse`) are ways to parse *one* date at a time.

`pandas` is generally oriented toward working with whole *arrays* of dates, whether used as an axis index or a column in a DataFrame. The `pandas.to_datetime` method parses many different kinds of date representations at once. Standard date formats like ISO 8601 can be parsed quickly:

In [18]:
datestrs = ["2011-07-06 12:00:00", "2011-08-06 00:00:00"]

pd.to_datetime(datestrs)

DatetimeIndex(['2011-07-06 12:00:00', '2011-08-06 00:00:00'], dtype='datetime64[us]', freq=None)

The result is a `DatetimeIndex` — pandas' array type specialized for timestamps (the next notebook covers it in depth). Note that `to_datetime` had to *infer* the format here by inspecting the strings, which has a real cost on large arrays with inconsistent formatting. If you already know the format, pass it explicitly with `format=` — this is both faster and removes any ambiguity about how to interpret the string:

```python
pd.to_datetime(datestrs, format="%Y-%m-%d %H:%M:%S")
```

`to_datetime` also handles values that should be considered missing (`None`, empty strings, etc.) by converting them to **`NaT`** ("Not a Time") — pandas' null value for datetime data, playing the same role `NaN` plays for floats. You'll see `NaT` throughout the rest of this chapter anywhere a timestamp is missing:

In [19]:
idx = pd.to_datetime(datestrs + [None])

idx

DatetimeIndex(['2011-07-06 12:00:00', '2011-08-06 00:00:00', 'NaT'], dtype='datetime64[us]', freq=None)

In [20]:
idx[2]

NaT

In [21]:
pd.isna(idx)

array([False, False,  True])

`datetime` objects also have a number of locale-specific formatting options for systems in other countries or languages. For example, abbreviated month names will be different on German or French systems compared with English systems.

| Type | Description |
|---|---|
| `%a` | Abbreviated weekday name |
| `%A` | Full weekday name |
| `%b` | Abbreviated month name |
| `%B` | Full month name |
| `%c` | Full date and time (e.g., `Tue 01 May 2012 04:20:57 PM`) |
| `%p` | Locale equivalent of AM or PM |
| `%x` | Locale-appropriate formatted date (e.g., in the United States, May 1, 2012 yields `05/01/2012`) |
| `%X` | Locale-appropriate time (e.g., `04:24:12 PM`) |

---

## Summary / Cheat Sheet

**Core `datetime` module types:** `date`, `time`, `datetime`, `timedelta`, `tzinfo` — pandas' own time series types build directly on these.

**Turning a string into a date, three ways:**

| Tool | Handles arrays? | Needs a format string? | Notes |
|---|---|---|---|
| `datetime.strptime(s, fmt)` | No, one at a time | Yes | Standard library, fastest when format is known |
| `dateutil.parser.parse(s)` | No, one at a time | No | Flexible/human-friendly, but slower and can guess wrong (`dayfirst=`) |
| `pandas.to_datetime(arr)` | Yes | Optional | The one you'll actually use for real datasets; pass `format=` when you know it |

**Nuances worth remembering:**
- `timedelta.seconds` is only the *leftover* seconds after whole days are removed — use `.total_seconds()` for the true total duration.
- Date parsing is ambiguous without more context: `6/12/2011` means very different things depending on locale (`dayfirst=True`/`False`).
- `NaT` is the datetime equivalent of `NaN` — pandas' way of representing a missing timestamp — and shows up throughout the rest of this chapter.

**Up next:** *Time Series Basics* — putting these building blocks into a pandas `Series`/`DataFrame` indexed by a `DatetimeIndex`, and what that unlocks for indexing and selection.